In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel,AutoTokenizer, AutoModel
import ast

In [ ]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/finallDataset.csv'
train_full = pd.read_csv(archivo_3)

# AST spliter D. Gries form

In [ ]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [ ]:
def DGries_states(solution):
  try:
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)




# Encoder Description and Code

In [ ]:
class Encoder:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.codebeart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings


  def tokenize_and_generate_embeddings_graphcodes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.graphcodebert_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.graphcodebert_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()
    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_graphcodebert_tokenizer(self):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.graphcodebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model = AutoModel.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model.to(device)


  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    #self.load_codebert_tokenizer()
    self.load_graphcodebert_tokenizer()
    self.is_loadtokenizers=True




# All characteristics

In [ ]:
def categoricallabelAll(w):
  if w=="['Correct']":
    return 0
  if w=="['Initial state']":
    return 1
  if w=="['Final state']":
    return 2
  if w=="['State transformation']":
    return 3
  if w=="['Initial state', 'Final state']":
    return 4
  if w=="['Initial state', 'State transformation']":
    return 5
  if w=="['Final state', 'State transformation']":
    return 6
  if w=="['Initial state', 'Final state', 'State transformation']":
    return 7
  return 8

category=np.array([
    'Correct',
    'Initial state',
    'Final state',
    'State transformation',
    'Initial state, Final state',
    'Initial state, State transformation',
    'Final state, State transformation',
    'Initial state, Final state, State transformation'

])

# Load Dataset

In [ ]:
train_full.head()

,Problema,Solución,y
0,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 1\n...,0
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,0
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,0
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,0
4,Write a Python function to check if a number i...,def is_prime(num):\n i = 2\n while i < n...,0


In [ ]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Solución'].apply(DGries_states).apply(pd.Series)

Streaming output truncated to the last 5000 lines.
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK c

In [ ]:
train_full

,Problema,Solución,y,Estado incial,Transformación de estado,Estado final
0,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 1\n...,0,result = 0\n i = 1,result += i\ni += 1,i <= n
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,0,total = 0\n i = 1,total += i\ni += 2,i < 100
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,0,numbers = []\n i = 0,print(i)\nnumbers[i] = i\ni += 1,i < 10
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,0,numbers = []\n i = 0,print(10 - i)\nnumbers.append(i)\ni += 1,i < 10
4,Write a Python function to check if a number i...,def is_prime(num):\n i = 2\n while i < n...,0,i = 2,if num % i == 0:\n return False\ni += 1,i < num
...,...,...,...,...,...,...
4130,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n i = 0\n cou...,6,i = 0\n count = 0,if i % 3 == 0:\n count = count + 1\ni = i + 1,i < n
4131,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n i = 0\n cou...,6,i = 0\n count = 0,if i % 3 == 0:\n count += 1\ni += 1,not i >= n
4132,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n count = 0\n ...,6,count = 0\n i = 0,if not i % 3:\n count += 1\ni += 1,n > i
4133,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n count = 0\n ...,6,count = 0\n i = 0,if i % 3 == 0 and True:\n count = count + 1...,i != n


In [ ]:
y=train_full['y'].to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [ ]:
train_full.dropna(inplace=True)

In [ ]:
train_full

,Problema,Solución,y,Estado incial,Transformación de estado,Estado final
0,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 1\n...,0,result = 0\n i = 1,result += i\ni += 1,i <= n
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,0,total = 0\n i = 1,total += i\ni += 2,i < 100
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,0,numbers = []\n i = 0,print(i)\nnumbers[i] = i\ni += 1,i < 10
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,0,numbers = []\n i = 0,print(10 - i)\nnumbers.append(i)\ni += 1,i < 10
4,Write a Python function to check if a number i...,def is_prime(num):\n i = 2\n while i < n...,0,i = 2,if num % i == 0:\n return False\ni += 1,i < num
...,...,...,...,...,...,...
4130,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n i = 0\n cou...,6,i = 0\n count = 0,if i % 3 == 0:\n count = count + 1\ni = i + 1,i < n
4131,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n i = 0\n cou...,6,i = 0\n count = 0,if i % 3 == 0:\n count += 1\ni += 1,not i >= n
4132,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n count = 0\n ...,6,count = 0\n i = 0,if not i % 3:\n count += 1\ni += 1,n > i
4133,calculate how many numbers are divisible by 3,def buggy_count_mult_3(n):\n count = 0\n ...,6,count = 0\n i = 0,if i % 3 == 0 and True:\n count = count + 1...,i != n


In [ ]:
encoder=Encoder()
encoder.start_tokenizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
%%time
problem=train_full['Problema'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
code=train_full['Solución'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

CPU times: user 2min 48s, sys: 2.52 s, total: 2min 50s
Wall time: 2min 48s


In [ ]:
problem.shape,code.shape,startstate.shape,finalstate.shape,transstate.shape

((4094,), (4094,), (4094,), (4094,), (4094,))

In [ ]:
print(startstate.shape)
print(startstate[1262].shape)


(4094,)
(1, 13, 768)


In [ ]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xcode=np.array([sentence[0].mean(axis=0) for sentence in code])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((4094, 768), (4094, 768), (4094, 768), (4094, 768))

In [ ]:
y=train_full['y'].to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [ ]:
y.shape

(4094,)

In [ ]:
from sklearn.model_selection import train_test_split

Xp_train,Xp_test,Xcode_train,Xcode_test,Xs_train,Xs_test,Xt_train,Xt_test,Xf_train,Xf_test,y_train,y_test=train_test_split(Xp,Xcode,Xs,Xt,Xf,y,test_size=0.2,random_state=2023, stratify=y)

In [ ]:
Xp_train.shape,Xp_test.shape,Xcode_train.shape, Xcode_test.shape, Xs_train.shape,Xs_test.shape,Xt_train.shape,Xt_test.shape,Xf_train.shape,Xf_test.shape,y_train.shape,y_test.shape

((3275, 768),
 (819, 768),
 (3275, 768),
 (819, 768),
 (3275, 768),
 (819, 768),
 (3275, 768),
 (819, 768),
 (3275, 768),
 (819, 768),
 (3275,),
 (819,))

In [ ]:
#random search
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import matthews_corrcoef
#XGBoost
from xgboost import XGBClassifier
#pipeline
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier


In [ ]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


# KNN

In [ ]:
from tensorflow.keras.losses import sparse_categorical_crossentropy
from sklearn.ensemble import VotingClassifier,StackingClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
accuracies_train=[]
accuracies_predict=[]
loss_predict=[]
times_train=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]
ml_names=[]
abl_names=[]

In [ ]:
import joblib

In [ ]:
path="/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/1. Tradicional Machine Learning/BestResult6/"

In [31]:
from time import time
knn=KNeighborsClassifier(n_neighbors=15,p=1,weights='distance')
svm_pipe=Pipeline([
    ('scaler',StandardScaler()),
    ('classifier',SVC(kernel='rbf',gamma=3.359818286283781e-05,C=69.51927961775606,probability=True))
])
mlp_pipe=Pipeline([
    ('scaler',StandardScaler()),
    ('classifier',MLPClassifier(hidden_layer_sizes=(100,100,100),learning_rate='adaptive'))
])

rf=RandomForestClassifier(n_estimators=80,max_depth=15)

vt=VotingClassifier(estimators=[('mlp',mlp_pipe),('rf',rf)],voting='soft')
st=StackingClassifier(estimators=[('mlp',mlp_pipe),('rf',rf)],final_estimator=LogisticRegression())

models=[knn,svm_pipe,mlp_pipe,rf,vt,st]
models_names=["KNN","SVM","MLP","Random Forest","Voting","Stacking"]
ablations_names=["problemall","problem_gries","code_gries","gries","problem_code","code"]

for model,name in zip(models,models_names):
  print(name)
  # Define the full datasets outside the inner loop
  X_train_full = np.concatenate((Xp_train,Xcode_train,Xs_train,Xt_train,Xf_train),axis=1)
  X_test_full = np.concatenate((Xp_test,Xcode_test,Xs_test,Xt_test,Xf_test),axis=1)

  for ablations_name in ablations_names:
    ml_names.append(name)
    abl_names.append(ablations_name)
    print(ablations_name)

    if ablations_name=="problemall":
      X_train = X_train_full
      X_test = X_test_full
    elif ablations_name=="problem_gries":
      X_train=np.concatenate((Xp_train,Xs_train,Xt_train,Xf_train),axis=1)
      X_test=np.concatenate((Xp_test,Xs_test,Xt_test,Xf_test),axis=1) # Corrected X_test construction
    elif ablations_name=="code_gries":
      X_train=np.concatenate((Xcode_train,Xs_train,Xt_train,Xf_train),axis=1)
      X_test=np.concatenate((Xcode_test,Xs_test,Xt_test,Xf_test),axis=1) # Corrected X_test construction
    elif ablations_name=="gries":
      X_train=np.concatenate((Xs_train,Xt_train,Xf_train),axis=1)
      X_test=np.concatenate((Xs_test,Xt_test,Xf_test),axis=1) # Corrected X_test construction
    elif ablations_name=="problem_code":
      X_train=np.concatenate((Xp_train,Xcode_train),axis=1)
      X_test=np.concatenate((Xp_test,Xcode_test),axis=1) # Corrected X_test construction
    elif ablations_name=="code":
      X_train=Xcode_train # Corrected X_train construction
      X_test=Xcode_test # Corrected X_test construction

    time1=time()
    model.fit(X_train,y_train)
    time2=time()
    joblib.dump(model,path+"graph_"+name+"_"+ablations_name+".joblib")
    times_train.append(time2-time1)
    time1=time()
    y_pred=model.predict(X_test)
    time2=time()
    times_predict.append(time2-time1)

    accuracie_train=model.score(X_train,y_train)
    accuracies_train.append(accuracie_train)
    accuracie_test=model.score(X_test,y_test)
    accuracies_predict.append(accuracie_test)
    #sparse_categorical_crossentropy
    y_pred_proba = model.predict_proba(X_test)
    loss=np.mean(sparse_categorical_crossentropy(y_test,y_pred_proba))
    loss_predict.append(loss)
    mcc = matthews_corrcoef(y_test, y_pred)
    mccs_predict.append(mcc)
    auc_pr = calculate_auc_pr_multiclass(y_test, model.predict_proba(X_test))
    aucpr_predict.append(auc_pr)
    print(classification_report(y_test,y_pred))
    print("Accuracy train",accuracie_train)
    print("Accuracy predict",accuracie_test)
    print("Loss",loss)
    print("MCC",mcc)
    print("AUC",auc_pr)
    print("==================================")

KNN
problemall
              precision    recall  f1-score   support

           0       0.94      0.97      0.96       208
           1       0.94      0.97      0.95       210
           2       0.96      0.93      0.94       281
           3       1.00      1.00      1.00        31
           4       0.96      0.80      0.87        30
           5       1.00      0.88      0.94        33
           6       0.93      1.00      0.96        26

    accuracy                           0.95       819
   macro avg       0.96      0.94      0.95       819
weighted avg       0.95      0.95      0.95       819

Accuracy train 0.9987786259541984
Accuracy predict 0.9487179487179487
Loss 0.3826905791312813
MCC 0.9314327240529273
AUC 0.9718385824902965
problem_gries
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       208
           1       0.94      0.98      0.96       210
           2       0.96      0.93      0.94       281
           3     

In [32]:
import pandas as pd
df=pd.DataFrame({
    "Model":ml_names,
    "Ablation":abl_names,
    "Accuracy_train":accuracies_train,
    "Accuracy_predict":accuracies_predict,
    "Loss":loss_predict,
    "Time_train":times_train,
    "Time_predict":times_predict,
    "MCC":mccs_predict,
    "AUC":aucpr_predict
})





In [33]:
df

,Model,Ablation,Accuracy_train,Accuracy_predict,Loss,Time_train,Time_predict,MCC,AUC
0,KNN,problemall,0.998779,0.948718,0.382691,0.006111,4.536839,0.931433,0.971839
1,KNN,problem_gries,0.998779,0.951160,0.383416,0.004442,3.547064,0.934719,0.972300
2,KNN,code_gries,0.998779,0.946276,0.367746,0.004646,3.547678,0.928159,0.973429
3,KNN,gries,0.998168,0.948718,0.368277,0.003758,2.582526,0.931456,0.972407
4,KNN,problem_code,0.998779,0.923077,0.416609,0.002913,1.777971,0.897594,0.963868
5,KNN,code,0.998779,0.935287,0.360728,0.001803,0.878540,0.914078,0.965721
6,SVM,problemall,0.966412,0.902320,0.340766,64.167769,5.635096,0.869353,0.914667
7,SVM,problem_gries,0.950229,0.875458,0.397162,49.501906,4.383139,0.833077,0.904307
8,SVM,code_gries,0.962748,0.890110,0.340713,46.498002,4.053945,0.853034,0.917339
9,SVM,gries,0.943511,0.880342,0.409085,33.798800,2.847913,0.839256,0.896718


In [34]:
path="/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/1. Tradicional Machine Learning/BestResult6/"

In [35]:
df.to_csv(path+"tradicionalML_ablationsResultsGraph.csv",index=False)

In [36]:
df=pd.read_csv(path+"tradicionalML_ablationsResultsGraph.csv")
df

,Model,Ablation,Accuracy_train,Accuracy_predict,Loss,Time_train,Time_predict,MCC,AUC
0,KNN,problemall,0.998779,0.948718,0.382691,0.006111,4.536839,0.931433,0.971839
1,KNN,problem_gries,0.998779,0.951160,0.383416,0.004442,3.547064,0.934719,0.972300
2,KNN,code_gries,0.998779,0.946276,0.367746,0.004646,3.547678,0.928159,0.973429
3,KNN,gries,0.998168,0.948718,0.368277,0.003758,2.582526,0.931456,0.972407
4,KNN,problem_code,0.998779,0.923077,0.416609,0.002913,1.777971,0.897594,0.963868
5,KNN,code,0.998779,0.935287,0.360728,0.001803,0.878540,0.914078,0.965721
6,SVM,problemall,0.966412,0.902320,0.340766,64.167769,5.635096,0.869353,0.914667
7,SVM,problem_gries,0.950229,0.875458,0.397162,49.501906,4.383139,0.833077,0.904307
8,SVM,code_gries,0.962748,0.890110,0.340713,46.498002,4.053945,0.853034,0.917339
9,SVM,gries,0.943511,0.880342,0.409085,33.798800,2.847913,0.839256,0.896718


# Externd Data

In [37]:
extern_file = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/Copia de ExternData.csv'
train_full = pd.read_csv(extern_file)
train_full

,Problem Description,Python Code,Source,Description Error,Error Label
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2
1,Sum numbers from 1 to n,def sum_to_n_buggy(n):\n total = 0\n i =...,Based on typical mistakes in Codeforces common...,i is incorrectly incremented,2
2,Prompt user for numbers until 0 is entered,def input_until_zero_buggy():\n num = int(i...,Common error from GeeksforGeeks Python while l...,number is never updated in the loop,2
3,Find first multiple of 7 greater than n,def first_multiple_7_buggy(n):\n i = n + 1\...,Typical error in CodeNet beginner loop problems,i not incremented,2
4,Calculate factorial of n,def factorial_buggy(n):\n result = 1\n w...,Inspired by classic factorial exercises,"decrement by 2, not 1",2
5,Count digits in a positive integer n,def count_digits_buggy(n):\n count = 0\n ...,Inspired by digit counting problems in Codefor...,n not updated inside the loop,2
6,Sum all even numbers up to n,def sum_evens_buggy(n):\n total = 0\n i ...,"CodeChef/Codeforces summing problems, conditio...",loop condition incorrect,1
7,Find a prime number using break,def find_prime_buggy():\n num = 2\n whil...,Inspired by break/continue loop errors from Py...,break outside the if block,2
8,Print cubes of numbers from 1 to n,def print_cubes_buggy(n):\n while i <= n:\n...,"Common initialization error, based on CodeNet ...",i not initialized,0
9,Prompt for password until correct,def prompt_password_buggy():\n password = i...,Typical validation loop error from Python begi...,password requested only once,2


In [38]:
train_full['Python Code']=train_full['Python Code'].apply(lambda x: x.replace("\\n","\n"))

In [39]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Python Code'].apply(DGries_states).apply(pd.Series)

OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body


In [40]:
train_full

,Problem Description,Python Code,Source,Description Error,Error Label,Estado incial,Transformación de estado,Estado final
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,i = 0\n count = 0,if i % 2 == 0:\n count += 1,i <= n
1,Sum numbers from 1 to n,def sum_to_n_buggy(n):\n total = 0\n i =...,Based on typical mistakes in Codeforces common...,i is incorrectly incremented,2,total = 0\n i = 1,total += i\ni = i + 0,i <= n
2,Prompt user for numbers until 0 is entered,def input_until_zero_buggy():\n num = int(i...,Common error from GeeksforGeeks Python while l...,number is never updated in the loop,2,num = int(input('Enter number (0 to exit): ')),"print('Number:', num)",num != 0
3,Find first multiple of 7 greater than n,def first_multiple_7_buggy(n):\n i = n + 1\...,Typical error in CodeNet beginner loop problems,i not incremented,2,i = n + 1,pass,i % 7 != 0
4,Calculate factorial of n,def factorial_buggy(n):\n result = 1\n w...,Inspired by classic factorial exercises,"decrement by 2, not 1",2,result = 1,result *= n\nn -= 2,n > 1
5,Count digits in a positive integer n,def count_digits_buggy(n):\n count = 0\n ...,Inspired by digit counting problems in Codefor...,n not updated inside the loop,2,count = 0,n % 10\ncount += 1,n > 0
6,Sum all even numbers up to n,def sum_evens_buggy(n):\n total = 0\n i ...,"CodeChef/Codeforces summing problems, conditio...",loop condition incorrect,1,total = 0\n i = 2,total += i\ni += 2,i < n
7,Find a prime number using break,def find_prime_buggy():\n num = 2\n whil...,Inspired by break/continue loop errors from Py...,break outside the if block,2,num = 2,is_prime = True\ndivisor = 2\nwhile divisor < ...,True
8,Print cubes of numbers from 1 to n,def print_cubes_buggy(n):\n while i <= n:\n...,"Common initialization error, based on CodeNet ...",i not initialized,0,print(i ** 3)\ni += 1,print(i ** 3)\ni += 1,i <= n
9,Prompt for password until correct,def prompt_password_buggy():\n password = i...,Typical validation loop error from Python begi...,password requested only once,2,password = input('Enter password: '),print('Incorrect'),password != 'secret'


In [41]:
train_full['Python Code']=train_full['Python Code'].apply(lambda x: x.replace("\\n","\n"))

In [42]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Python Code'].apply(DGries_states).apply(pd.Series)

OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body


In [43]:
%%time
problem=train_full['Problem Description'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
code=train_full['Python Code'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()

CPU times: user 1.39 s, sys: 1.01 ms, total: 1.39 s
Wall time: 1.39 s


In [44]:
problem.shape,code.shape,startstate.shape,finalstate.shape,transstate.shape

((35,), (35,), (35,), (35,), (35,))

In [45]:
print(startstate.shape)
print(startstate[0].shape)


(35,)
(1, 12, 768)


In [46]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xcode=np.array([sentence[0].mean(axis=0) for sentence in code])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((35, 768), (35, 768), (35, 768), (35, 768))

In [47]:
y=train_full['Error Label']
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [48]:
accuracies_predict=[]
loss_predict=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]
ml_names=[]
abl_names=[]

ml_names=[]
abl_names=[]

In [49]:
from time import time


models_names=["KNN","SVM","MLP","Random Forest","Voting","Stacking"]
ablations_names=["problemall","problem_gries","code_gries","gries","problem_code","code"]

for name in models_names:
  print(name)
  for ablations_name in ablations_names:
    ml_names.append(name)
    abl_names.append(ablations_name)
    print(ablations_name)
    model=joblib.load(path+"graph_"+name+"_"+ablations_name+".joblib")
    print(model)
    if ablations_name=="problemall":
      Xe= np.concatenate((Xp,Xcode,Xs,Xt,Xf),axis=1)
    elif ablations_name=="problem_gries":
      Xe=np.concatenate((Xp,Xs,Xt,Xf),axis=1)
    elif ablations_name=="code_gries":
      Xe=np.concatenate((Xcode,Xs,Xt,Xf),axis=1)
    elif ablations_name=="gries":
      Xe=np.concatenate((Xs,Xt,Xf),axis=1)
    elif ablations_name=="problem_code":
      Xe=np.concatenate((Xp,Xcode),axis=1)
    elif ablations_name=="code":
      Xe=Xcode

    time1=time()
    y_pred=model.predict(Xe)
    time2=time()
    times_predict.append(time2-time1)

    accuracie_test=model.score(Xe,y)
    accuracies_predict.append(accuracie_test)
    #sparse_categorical_crossentropy
    y_pred_proba = model.predict_proba(Xe)
    loss=np.mean(sparse_categorical_crossentropy(y,y_pred_proba))
    loss_predict.append(loss)
    mcc = matthews_corrcoef(y, y_pred)
    mccs_predict.append(mcc)
    auc_pr = calculate_auc_pr_multiclass(y, y_pred_proba)
    aucpr_predict.append(auc_pr)
    print(classification_report(y,y_pred))
    print("Accuracy predict",accuracie_test)
    print("Loss",loss)
    print("MCC",mcc)
    print("AUC",auc_pr)
    print("==================================")

KNN
problemall
KNeighborsClassifier(n_neighbors=15, p=1, weights='distance')
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      0.40      0.57         5
           2       0.85      0.96      0.90        24
           3       1.00      1.00      1.00         1
           4       1.00      1.00      1.00         1
           5       1.00      1.00      1.00         1
           6       0.00      0.00      0.00         1

    accuracy                           0.86        35
   macro avg       0.84      0.77      0.78        35
weighted avg       0.87      0.86      0.84        35

Accuracy predict 0.8571428571428571
Loss 0.2600383573087797
MCC 0.7032636776948774
AUC 0.8736767606842838
problem_gries
KNeighborsClassifier(n_neighbors=15, p=1, weights='distance')
              precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       1.00      0.40  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.50      1.00      0.67         2
           1       0.67      0.40      0.50         5
           2       0.84      0.88      0.86        24
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       0.33      1.00      0.50         1
           6       0.00      0.00      0.00         1

    accuracy                           0.74        35
   macro avg       0.33      0.47      0.36        35
weighted avg       0.71      0.74      0.71        35

Accuracy predict 0.7428571428571429
Loss 1.2026496834969556
MCC 0.48097197110408335
AUC 0.8338160842293906
code
KNeighborsClassifier(n_neighbors=15, p=1, weights='distance')
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       0.75      0.60      0.67         5
           2       0.96      0.96      0.96        24
           3

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.50      1.00      0.67         2
           1       1.00      0.40      0.57         5
           2       0.82      0.96      0.88        24
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       1.00      1.00      1.00         1
           6       0.00      0.00      0.00         1

    accuracy                           0.80        35
   macro avg       0.47      0.48      0.45        35
weighted avg       0.76      0.80      0.75        35

Accuracy predict 0.8
Loss 0.5351975409733363
MCC 0.5681759430741397
AUC 0.7584075799687743
problem_gries
Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 SVC(C=69.51927961775606, gamma=3.359818286283781e-05,
                     probability=True))])


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      0.40      0.57         5
           2       0.80      1.00      0.89        24
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       1.00      1.00      1.00         1
           6       0.00      0.00      0.00         1

    accuracy                           0.83        35
   macro avg       0.54      0.49      0.49        35
weighted avg       0.78      0.83      0.78        35

Accuracy predict 0.8285714285714286
Loss 0.6310691176314871
MCC 0.6346351669793114
AUC 0.8387736911552904
code_gries
StackingClassifier(estimators=[('mlp',
                                Pipeline(steps=[('scaler', StandardScaler()),
                                                ('classifier',
                                                 MLPClassifier(hidden_layer_sizes=(100,
                  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      0.60      0.75         5
           2       0.80      1.00      0.89        24
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1

    accuracy                           0.83        35
   macro avg       0.40      0.37      0.38        35
weighted avg       0.75      0.83      0.77        35

Accuracy predict 0.8285714285714286
Loss 0.6479507033495802
MCC 0.6295662366690628
AUC 0.8739554011259676
code
StackingClassifier(estimators=[('mlp',
                                Pipeline(steps=[('scaler', StandardScaler()),
                                                ('classifier',
                                                 MLPClassifier(hidden_layer_sizes=(100,
                        

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [50]:
import pandas as pd
dfe=pd.DataFrame({
    "Model":ml_names,
    "Ablation":abl_names,
    "Accuracy_predict":accuracies_predict,
    "Loss":loss_predict,
    "Time_predict":times_predict,
    "MCC":mccs_predict,
    "AUC":aucpr_predict
})





In [51]:
dfe

,Model,Ablation,Accuracy_predict,Loss,Time_predict,MCC,AUC
0,KNN,problemall,0.857143,0.260038,0.186745,0.703264,0.873677
1,KNN,problem_gries,0.857143,0.314491,0.148788,0.701740,0.852067
2,KNN,code_gries,0.885714,0.303079,0.149793,0.777944,0.878333
3,KNN,gries,0.857143,0.765357,0.111068,0.729047,0.800935
4,KNN,problem_code,0.742857,1.202650,0.075899,0.480972,0.833816
5,KNN,code,0.914286,0.207484,0.039625,0.831720,0.979960
6,SVM,problemall,0.800000,0.535198,0.235237,0.568176,0.758408
7,SVM,problem_gries,0.771429,0.635576,0.202938,0.502206,0.744950
8,SVM,code_gries,0.857143,0.396415,0.191295,0.704349,0.993033
9,SVM,gries,0.828571,0.524383,0.131582,0.653238,0.853177


In [52]:
dfe.to_csv(path+"graph_externData_ablationsResults.csv",index=False)

In [53]:
dfe=pd.read_csv(path+"graph_externData_ablationsResults.csv")
dfe

,Model,Ablation,Accuracy_predict,Loss,Time_predict,MCC,AUC
0,KNN,problemall,0.857143,0.260038,0.186745,0.703264,0.873677
1,KNN,problem_gries,0.857143,0.314491,0.148788,0.701740,0.852067
2,KNN,code_gries,0.885714,0.303079,0.149793,0.777944,0.878333
3,KNN,gries,0.857143,0.765357,0.111068,0.729047,0.800935
4,KNN,problem_code,0.742857,1.202650,0.075899,0.480972,0.833816
5,KNN,code,0.914286,0.207484,0.039625,0.831720,0.979960
6,SVM,problemall,0.800000,0.535198,0.235237,0.568176,0.758408
7,SVM,problem_gries,0.771429,0.635576,0.202938,0.502206,0.744950
8,SVM,code_gries,0.857143,0.396415,0.191295,0.704349,0.993033
9,SVM,gries,0.828571,0.524383,0.131582,0.653238,0.853177
